In [ ]:
# ============================================================
# Alpha-only Pareto Front Experiment (MULTI-ALPHA)
# Convex / Concave PF via FLAG
# SS vs PS — Intersections, HV, IGD, IGD+
# PF defined by:
#   convex : f = 1 - U / ||U||_alpha
#   concave: f = U / ||U||_alpha
# ============================================================
#Import packages
import os
import numpy as np
from numba import njit, prange
import itertools
import math
from pathlib import Path
import plotly.graph_objects as go
import matplotlib.pyplot as plt

#Import utilities
from Codes.PHS_sampler import phs_sampler
from Codes.hv_qmc import Compute_HV

# ============================================================
#  Output directories
# ============================================================
BASE_DIR = Path.cwd()
FIG_DIR = BASE_DIR /"Results/Figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = BASE_DIR / "Results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
#  Das–Dennis
# ============================================================
def das_dennis(M, H):
    slots = H + M - 1
    pts = []
    for c in itertools.combinations(range(slots), M - 1):
        prev = -1
        counts = []
        for pos in c:
            counts.append(pos - prev - 1)
            prev = pos
        counts.append(slots - prev - 1)
        w = np.array(counts, float) / H
        w /= w.sum()
        pts.append(w)
    return np.vstack(pts)


# ============================================================
#  Shift to sum=M hyperplane
# ============================================================
def shift_to_hyperplane(pts, target_sum):
    pts = np.asarray(pts, float)
    alphas = (target_sum - pts.sum(axis=1)) / pts.shape[1]
    return pts + alphas[:, None]


# ============================================================
#  Uniform TRUE PF sampler (alpha + convex/concave) - Pareto Reference Set
# ============================================================
def pf_size_for_M(M):
    if M == 3:
        return 2000
    if M == 5:
        return 5000
    if M == 8:
        return 15000
    if M == 10:
        return 50000
    if M == 15:
        return 80000
    return 20000

def dd_H_at_most(M, N_target):
    H = 1
    while True:
        if math.comb(H + M - 1, M - 1) > N_target:
            return H - 1
        H += 1


def sample_true_pf_alpha(M, alpha, problem_type, seed=0):
    """
    Uniform PF for IGD / IGD+
    """
    N = pf_size_for_M(M)
    U = das_dennis(M, H=dd_H_at_most(M, N))
    den = np.power(np.sum(U**alpha, axis=1), 1.0 / alpha)
    mask = den > 1e-12
    U = U[mask]
    den = den[mask]

    if problem_type == "convex":
        PF = 1.0 - U / den[:, None]
    elif problem_type == "concave":
        PF = U / den[:, None]
    else:
        raise ValueError("problem_type must be 'convex' or 'concave'")

    return np.clip(PF, 0.0, 1.0)

# ============================================================
#  Ray–PF intersection (implicit PF)
# ============================================================
def intersect_alpha_pf(P, M, alpha, problem_type, max_iter=100, tol=1e-12):
    """
    Robust ray–PF intersection for alpha-only PFs.

    Ray: F(t) = P - t * 1
    Domain: 0 <= F <= 1

    concave PF : sum(f_i^alpha) = 1
    convex  PF : sum((1 - f_i)^alpha) = 1
    """
    P = np.asarray(P, float)
    N, M = P.shape

    # Feasible ray interval inside [0,1]^M
    t_lo = np.maximum(0.0, np.max(P - 1.0, axis=1))
    t_hi = np.min(P, axis=1)

    active = t_lo <= t_hi
    if not np.any(active):
        return np.empty((0, M))

    P = P[active]
    t_lo = t_lo[active]
    t_hi = t_hi[active]

    # --------------------------------------------------------
    # Monotone PF function along ray
    # --------------------------------------------------------
    def phi(t):
        F = P - t[:, None]
        valid = (F >= 0).all(axis=1) & (F <= 1).all(axis=1)
        out = np.full(len(t), np.nan)

        if np.any(valid):
            if problem_type == "concave":
                out[valid] = np.sum(F[valid] ** alpha, axis=1) - 1.0
            elif problem_type == "convex":
                out[valid] = np.sum((1.0 - F[valid]) ** alpha, axis=1) - 1.0
            else:
                raise ValueError("problem_type must be 'convex' or 'concave'")
        return out

    f_lo = phi(t_lo)
    f_hi = phi(t_hi)

    # --------------------------------------------------------
    # Correct bracketing
    # concave : phi decreases, want f_lo >= 0 >= f_hi
    # convex  : phi increases, want f_lo <= 0 <= f_hi
    # --------------------------------------------------------
    if problem_type == "concave":
        mask = (
            np.isfinite(f_lo) & np.isfinite(f_hi) &
            (f_lo >= -tol) & (f_hi <= tol)
        )
    else:  # convex
        mask = (
            np.isfinite(f_lo) & np.isfinite(f_hi) &
            (f_lo <= tol) & (f_hi >= -tol)
        )

    if not np.any(mask):
        return np.empty((0, M))

    P = P[mask]
    t_lo = t_lo[mask]
    t_hi = t_hi[mask]
    f_lo = f_lo[mask]
    f_hi = f_hi[mask]

    # --------------------------------------------------------
    # Vectorized bisection
    # --------------------------------------------------------
    for _ in range(max_iter):
        t_mid = 0.5 * (t_lo + t_hi)
        f_mid = phi(t_mid)
        finite = np.isfinite(f_mid)

        if problem_type == "concave":
            go_right = finite & (f_mid > 0)
        else:  # convex
            go_right = finite & (f_mid < 0)

        t_lo[go_right] = t_mid[go_right]
        f_lo[go_right] = f_mid[go_right]

        t_hi[~go_right] = t_mid[~go_right]
        f_hi[~go_right] = f_mid[~go_right]

    t = 0.5 * (t_lo + t_hi)
    X = P - t[:, None]
    return np.clip(X, 0.0, 1.0)

# ============================================================
#  Metrics - HV, IGD, IGD+, Coverage error
# ============================================================
def safe_hv(A):
    if len(A) == 0:
        return float("nan")
    return float(Compute_HV(A))

@njit(parallel=True, fastmath=True)
def _min_dists_igd(A, P):
    nP, M = P.shape
    nA = A.shape[0]
    out = np.empty(nP, dtype=np.float64)

    for i in prange(nP):
        min_d2 = 1e300
        for j in range(nA):
            d2 = 0.0
            for k in range(M):
                diff = P[i, k] - A[j, k]
                d2 += diff * diff
            if d2 < min_d2:
                min_d2 = d2
        out[i] = np.sqrt(min_d2)
    return out

def igd(A, P):
    if len(A) == 0 or len(P) == 0:
        return float("nan")
    A = np.asarray(A, dtype=np.float64)
    P = np.asarray(P, dtype=np.float64)
    mins = _min_dists_igd(A, P)
    return float(mins.mean())



@njit(parallel=True, fastmath=True)
def _min_dists_igd_plus(A, P):
    nP, M = P.shape
    nA = A.shape[0]
    out = np.empty(nP, dtype=np.float64)

    for i in prange(nP):
        min_d2 = 1e300
        for j in range(nA):
            d2 = 0.0
            for k in range(M):
                diff = A[j, k] - P[i, k]
                if diff > 0.0:
                    d2 += diff * diff
            if d2 < min_d2:
                min_d2 = d2
        out[i] = np.sqrt(min_d2)
    return out

def igd_plus(A, P):
    if len(A) == 0 or len(P) == 0:
        return float("nan")
    A = np.asarray(A, dtype=np.float64)
    P = np.asarray(P, dtype=np.float64)
    mins = _min_dists_igd_plus(A, P)
    return float(mins.mean())




@njit(parallel=True, fastmath=True)
def _min_dists_for_coverage(A, P):
    nP, M = P.shape
    nA = A.shape[0]
    out = np.empty(nP, dtype=np.float64)

    for i in prange(nP):
        min_d2 = 1e300
        for j in range(nA):
            d2 = 0.0
            for k in range(M):
                diff = P[i, k] - A[j, k]
                d2 += diff * diff
            if d2 < min_d2:
                min_d2 = d2
        out[i] = np.sqrt(min_d2)
    return out

def coverage_error(A, P):
    if len(A) == 0 or len(P) == 0:
        return float("nan")
    A = np.asarray(A, dtype=np.float64)
    P = np.asarray(P, dtype=np.float64)
    mins = _min_dists_for_coverage(A, P)
    return float(mins.max())

# ============================================================
#  Plotting (M=3)
# ============================================================
def plot_3d(PF, P_ref, I_pts, title):
    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=PF[:,0], y=PF[:,1], z=PF[:,2],
        mode="markers",
        marker=dict(size=4, opacity=0.15, color="blue"),
        name="True PF"
    ))

    fig.add_trace(go.Scatter3d(
        x=P_ref[:,0], y=P_ref[:,1], z=P_ref[:,2],
        mode="markers",
        marker=dict(size=4, color="red"),
        name="Refs"
    ))

    if len(I_pts):
        fig.add_trace(go.Scatter3d(
            x=I_pts[:,0], y=I_pts[:,1], z=I_pts[:,2],
            mode="markers",
            marker=dict(size=6, color="green"),
            name="Intersections"
        ))

    fig.update_layout(title=title)
    fig.update_layout(
        scene=dict(
            aspectmode="cube",
            xaxis=dict(
                title=dict(text="F1", font=dict(size=22)),
                tickfont=dict(size=16),
            ),
            yaxis=dict(
                title=dict(text="F2", font=dict(size=22)),
                tickfont=dict(size=16),
            ),
            zaxis=dict(
                title=dict(text="F3", font=dict(size=22)),
                tickfont=dict(size=16),
            ),
        ),
        margin=dict(l=0, r=0, t=0, b=0),
        width=900,
        height=800,
    )

    fig.show()

def save_matplotlib_3d(PF, P_ref, I_pts, title, fname):

    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(*PF.T, s=17, alpha=0.35, c="blue", marker="o", label = "True PF")
    # ax.scatter(*P_ref.T, alpha=0.75, s=18, c="red", label="Refs") 
    if len(I_pts):
        ax.scatter(*I_pts.T, s=40, c="green",  marker="o", label="Intersections")

    ax.view_init(elev=25, azim=45)
    ax.set_box_aspect((1, 1, 1))
    ax.margins(0)
    plt.subplots_adjust(left=0.1, right=1, bottom=0.05, top=1)
    ax.set_aspect('equal')
    # Axis labels + tick size
    ax.set_xlabel(r'$f1$', fontsize=15, labelpad=6)
    ax.set_ylabel(r'$f2$', fontsize=15, labelpad=6)
    ax.set_zlabel(r'$f3$', fontsize=15, labelpad=6)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.grid(False)
    out = FIG_DIR / fname
    fig.savefig(out, dpi=300, bbox_inches="tight", pad_inches=0.2)
    plt.close(fig)

    print(f"[Saved] {out}")

# ============================================================
#  Plotting (M>3) - Higher dimensions
# ============================================================

def parallel_axis_plot(samples,
                       nlines=300,
                       normalize=False,
                       color_by=None,
                       axis_labels=None,
                       alpha=0.5,
                       linewidth=0.8,
                       cmap="viridis",
                       figsize=(12, 6),
                       seed=0,
                       title=None,
                       show=True,
                       save_figure=False,
                       save_path="Results"):
    """
    Parallel-axis (parallel-coordinates) plot for samples.

    Parameters
    ----------
    samples : ndarray, shape (n_samples, N)
        Input samples (RAW coordinates, e.g. loaded from .npy).
    nlines : int
        Number of sample lines to plot (random subset if samples are many).
    normalize : bool
        If True, normalize each axis independently to [0,1].
        If False, plot raw coordinates.
    color_by : None or int
        If None, all lines are same color.
        If int k, color lines by value on axis k.
    axis_labels : list of str or None
        Labels for axes. Defaults to x0, x1, ..., x{N-1}.
    """

    samples = np.asarray(samples)
    if samples.ndim != 2:
        raise ValueError("samples must be a 2D array (n_samples, N)")

    n_samples, N = samples.shape

    # subsample lines
    rng = np.random.default_rng(seed)
    if n_samples > nlines:
        idx = rng.choice(n_samples, size=nlines, replace=False)
        X = samples[idx]
    else:
        X = samples.copy()

    # normalization (optional)
    if normalize:
        mn = X.min(axis=0)
        mx = X.max(axis=0)
        scale = mx - mn
        scale[scale == 0.0] = 1.0
        Xp = (X - mn) / scale
        y_label = "Normalized value"
    else:
        Xp = X
        y_label = "Raw coordinate value"

    # axis labels
    if axis_labels is None:
        axis_labels = [f"x{i}" for i in range(N)]

    xs = np.arange(N)

    # color mapping
    if color_by is None:
        colors = [(0, 0, 0, alpha)] * Xp.shape[0]
    else:
        if not (0 <= color_by < N):
            raise ValueError("color_by must be None or an axis index in [0, N-1]")
        vals = Xp[:, color_by]
        vmin, vmax = vals.min(), vals.max()
        if vmax > vmin:
            vals = (vals - vmin) / (vmax - vmin)
        cmap_obj = plt.get_cmap(cmap)
        colors = [cmap_obj(v) for v in vals]

    # plotting
    fig, ax = plt.subplots(figsize=figsize)
    for i in range(Xp.shape[0]):
        ax.plot(xs, Xp[i], color=colors[i], alpha=alpha, linewidth=linewidth)

    # vertical axes
    for xi in xs:
        ax.axvline(xi, color="k", linewidth=0.8, alpha=0.5)

    ax.set_xlim(xs[0], xs[-1])
    ax.set_xticks(xs)
    ax.set_xticklabels(axis_labels, rotation=45, ha="right")
    ax.set_ylabel(y_label)

    if title:
        ax.set_title(title)

    plt.tight_layout()

    if save_figure:
        plt.savefig(save_path, dpi=300)

    if show:
        plt.show()

    return fig, ax

# ============================================================
#  main
# ============================================================
def run(
    M=3,
    alphas=(2, 4, 8),
    problem_type="convex",   # "convex" or "concave"
    samplers=("SS", "PHS"),
    H_list=(15,),
    seed=0,
    save_figures=True,
):
    
    results_path = RESULTS_DIR / f"Results_{problem_type}_M{M}_seed_{seed}.txt"

    # ------------------------------------------------------------------
    # Cache directories
    # ------------------------------------------------------------------
    ref_root = RESULTS_DIR / "Ref_points"
    int_root = RESULTS_DIR / "Intersections"

    for sampler in samplers:
        (ref_root / sampler).mkdir(parents=True, exist_ok=True)
        (int_root / sampler).mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------------
    # Main experiment
    # ------------------------------------------------------------------
    with open(results_path, "w") as f:
        f.write("============================================================\n")
        f.write(f"Alpha-only PF Experiment | type={problem_type}\n")
        f.write(f"M={M}, alphas={list(alphas)}, seed={seed}\n")
        f.write("============================================================\n\n")

        for alpha in alphas:
            f.write(f"\n##############################\n")
            f.write(f"Problem: alpha = {alpha}\n")
            f.write(f"##############################\n\n")

            PF_true = sample_true_pf_alpha(
                M, alpha, problem_type, seed=seed
            )

            for H in H_list:
                f.write(f"--- H={H} ---\n")

                for sampler in samplers:

                    # ==================================================
                    # Reference points 
                    # ==================================================
                    if sampler == "SS":
                        ref_file = (
                            ref_root / sampler /
                            f"ref_M{M}_H{H}.npy"
                        )
                    else:  # PHS 
                        ref_file = (
                            ref_root / sampler /
                            f"ref_M{M}_H{H}_seed{seed}.npy"
                        )

                    if ref_file.exists():
                        P_ref = np.load(ref_file)
                    else:
                        if sampler == "SS":
                            P_ref = das_dennis(M, H)
                        else:
                            N_ref = math.comb(H + M - 1, M - 1)
                            P_ref = phs_sampler(
                                d=M,
                                N_target=N_ref,
                                seed=seed
                            )

                        P_ref = shift_to_hyperplane(
                            P_ref, target_sum=M
                        )
                        np.save(ref_file, P_ref)

                    # ==================================================
                    # Intersection points
                    # ==================================================
                    if sampler == "SS":
                        int_file = (
                            int_root / sampler /
                            f"int_{problem_type}_M{M}_alpha{alpha}_H{H}.npy"
                        )
                    else:
                        int_file = (
                            int_root / sampler /
                            f"int_{problem_type}_M{M}_alpha{alpha}_H{H}_seed{seed}.npy"
                        )

                    if int_file.exists():
                        I_pts = np.load(int_file)
                    else:
                        I_pts = intersect_alpha_pf(
                            P_ref, M, alpha, problem_type
                        )
                        np.save(int_file, I_pts)

                    # ==================================================
                    # Metrics
                    # ==================================================
                    hv = safe_hv(I_pts)
                    igd_v = igd(I_pts, PF_true)
                    igdp_v = igd_plus(I_pts, PF_true)
                    cov_v = coverage_error(I_pts, PF_true)


                    print("================================")
                    print(f"type={problem_type} | alpha={alpha} | {sampler} | H={H}")
                    print(f"Refs={len(P_ref)} Hits={len(I_pts)} PF_True={len(PF_true)}")
                    print(f"HV={hv:.6f} IGD={igd_v:.6f} IGD+={igdp_v:.6f} COV={cov_v:.6f}")
                    print("================================")

                    f.write(
                        f"{sampler:2s} | refs={len(P_ref):6d} | hits={len(I_pts):6d} | TruePF={len(PF_true):6d} | "
                        f"HV={hv:.10f} | IGD={igd_v:.10f} | IGD+={igdp_v:.10f} | "
                        f"COV={cov_v:.10f}\n"
                    )

                    # ==================================================
                    # Plotting (unchanged)
                    # ==================================================
                    title = f"{problem_type} | α={alpha} | {sampler} | M={M} H={H}"

                    if save_figures:
                        if M == 3:
                            plot_3d(PF_true, P_ref, I_pts, title)

                            save_matplotlib_3d(
                                PF_true,
                                P_ref,
                                I_pts,
                                title,
                                fname=f"{problem_type}_alpha{alpha}_{sampler}_M{M}_H{H}"
                            )
                        else:
                            parallel_axis_plot(
                                P_ref,
                                title=f"Refs | {title}"
                            )
                            if len(I_pts):
                                parallel_axis_plot(
                                    I_pts,
                                    title=f"Intersections | {title}",
                                    save_figure=False,
                                    save_path=FIG_DIR / f"{problem_type}_alpha{alpha}_{sampler}_M{M}_H{H}_intersections.png"
                                )

                f.write("\n")

        f.write("============================================================\n")

    print(f"\n[Saved results] {results_path}")
    return P_ref, I_pts

# ============================================================
#  RUN EXPERIMENTS
# ============================================================
if __name__ == "__main__":
    # ============================================================
    #Parameter settings for the experiments. You can adjust M, alphas, problem_type, samplers, H_list, seed, and save_figures as needed.
        # M = 3, 5, 8, 10, 15
        # alphas = [1, 2, 5, 10, 15, 20, 25, 30]
        #samplers = ["SS", "PHS"]
        # H_list = [12, 15, 20, 25, 30] <<< M=3
        # H_list = [6, 8, 9, 10, 11] <<< M=5
        # H_list = [4, 5, 6, 7, 8] <<< M=8
        # H_list = [4, 5, 6, 7, 8] <<< M=10
        # H_list = [3, 4, 5, 6] <<< M=15
    #ref_point and intersection points are saved and retrived for plotting purposes. If you don't want save intersection file you can modify.
    # PHS parameters can be modified in the PHS_sampler python code in the Codes folder
    # Experiments
    # ============================================================
    run(
        M=3,
        alphas=[1],
        problem_type="concave",   #"convex" or "concave"
        samplers=["SS", "PHS"],
        H_list=[12,15,20],
        seed=0,
        save_figures=True,
    )
    # for seed in range(1,32):
    # PF experiments
    # # ============================================================
    #     run(
    #         M=3,
    #         alphas=[1, 10, 20, 30],
    #         problem_type="convex",   #"convex" or "concave"
    #         samplers=["SS", "PHS"],
    #         H_list=[12, 15, 20, 25, 30],
    #         seed=seed,
    #         save_figures=False,
    #     )
    

In [ ]:
##################################################################
'''IGD and Coverage error are used jointly to assess convergence 
accuracy and Pareto-front coverage quality. A lower IGD value 
indicates that the approximated front is closer to the true Pareto
front, while a lower coverage error indicates better coverage
of the true Pareto front by the approximated front.'''

'''It read the results from the text files generated by the experiments 
from the Result folder, extracts the relevant data, and creates plots to
visualize the performance of the samplers (SS and PS) across different 
alpha values and reference point counts. The plots will show how the IGD 
and Coverage error metrics vary with the number of reference points for 
each sampler and alpha value, allowing for a comparative analysis of 
their performance in approximating the Pareto front.'''

##################################################################
#Plot results
import re
import math
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIG — CHANGE ONLY THESE
# ============================================================
RESULTS_DIR = Path("Results")
FIG_DIR = Path("Results/Figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

M_LIST = [3]    # <<< [3, 5, 8, 10]
PROBLEM_TYPES = ["convex","concave"]

METRIC = "IGD"      # <<< "IGD", "COV"
COLS_PER_ROW = 3

COLORS = {"SS": "red", "PHS": "green"}
MARKERS = {"SS": "o", "PHS": "s"}

# ============================================================
# REGEX (extended to include Coverage Error)
# ============================================================
re_alpha = re.compile(r"^Problem:\s*alpha\s*=\s*([0-9.eE+-]+)\s*$")
re_row = re.compile(
    r"^(SS|PHS)\s*\|\s*refs=\s*(\d+)\s*\|\s*hits=\s*(\d+)\s*\|\s*"
    r"TruePF=\s*(\d+)\s*\|\s*"
    r"HV=([0-9.eE+-]+)\s*\|\s*"
    r"IGD=([0-9.eE+-]+)\s*\|\s*"
    r"IGD\+=([0-9.eE+-]+)\s*\|\s*"
    r"COV=([0-9.eE+-]+)\s*$"
)

# ============================================================
# PARSER
# ============================================================
def parse_results_alpha(M_target: int, problem_type: str) -> pd.DataFrame:
    fp = RESULTS_DIR / f"Results_{problem_type}_M{M_target}_seed_0.txt"
    if not fp.exists():
        raise FileNotFoundError(fp)

    rows = []
    current_alpha = None

    for raw in fp.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line:
            continue

        ma = re_alpha.match(line)
        if ma:
            current_alpha = float(ma.group(1))
            continue

        mr = re_row.match(line)
        if mr and current_alpha is not None:
            rows.append({
                "M": M_target,
                "problem_type": problem_type,
                "alpha": current_alpha,
                "sampler": mr.group(1),
                "refs": int(mr.group(2)),
                "hits": int(mr.group(3)),
                "TruePF": int(mr.group(4)),
                "HV": float(mr.group(5)),
                "IGD": float(mr.group(6)),
                "IGD+": float(mr.group(7)),
                "COV": float(mr.group(8)),
            })


    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No data parsed from {fp.name}")

    # sanity check
    dup = df.duplicated(subset=["alpha", "sampler", "refs"], keep=False)
    if dup.any():
        raise RuntimeError(
            f"Duplicate entries detected in {fp.name}:\n{df[dup]}"
        )

    return df

# ============================================================
# PLOTTING
# ============================================================
def plot_metric_by_alpha(
    df: pd.DataFrame,
    M_target: int,
    problem_type: str,
    metric: str,
    cols: int = 3,
):
    if metric not in {"HV", "IGD", "IGD+", "COV"}:
        raise ValueError("metric must be 'HV', 'IGD', 'IGD+', or 'COV'")

    alphas = sorted(df["alpha"].unique())
    n = len(alphas)
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(5.6 * cols, 4.4 * rows),
        sharey=False
    )

    # normalize axes indexing
    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]

    for i, alpha in enumerate(alphas):
        r = i // cols
        c = i % cols
        ax = axes[r][c]

        d = df[df["alpha"] == alpha]

        for sampler in ["SS", "PHS"]:
            ds = d[d["sampler"] == sampler].sort_values("refs")
            if ds.empty:
                continue

            ax.plot(
                ds["refs"],
                ds[metric],
                color=COLORS[sampler],
                marker=MARKERS[sampler],
                linewidth=2,
                markersize=6,
                label=sampler
            )

        ax.set_title(f"alpha = {alpha:g}")
        ax.set_xlabel("Reference points")
        ax.set_ylabel(metric)
        ax.grid(True, linestyle=":")
        ax.legend(frameon=False)

    # disable unused axes
    for j in range(n, rows * cols):
        r = j // cols
        c = j % cols
        axes[r][c].axis("off")

    fig.suptitle(
        f"{metric} vs reference points | M={M_target} | type={problem_type}",
        fontsize=14
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    out = FIG_DIR / f"{metric}_grid_M{M_target}_{problem_type}.png"
    fig.savefig(out, dpi=300)
    print(f"[Saved] {out}")

    plt.show()

# ============================================================
# MAIN
# ============================================================
for M in M_LIST:
    for problem_type in PROBLEM_TYPES:
        try:
            df = parse_results_alpha(M, problem_type)
            plot_metric_by_alpha(
                df,
                M_target=M,
                problem_type=problem_type,
                metric=METRIC,
                cols=COLS_PER_ROW,
            )
        except Exception as e:
            print(f"[Skip] M={M}, type={problem_type}: {e}")


In [ ]:
# Save the result in csv for all M 
#======================================================================================
'''This section reads the results from the text files from the Results folder, extracts 
the relevant data using regex, and compiles it into a structured format (DataFrame). 
It then builds a combined tablefor the specified metric (IGD or COV) across all M values
and saves it as both CSV and Excel files for further analysis or reporting.'''
#======================================================================================
import re
from pathlib import Path
import pandas as pd

# ============================================================
# CONFIG — CHANGE ONLY THESE
# ============================================================
RESULTS_DIR = Path("Results")
OUT_DIR = Path("Results/Tables")
OUT_DIR.mkdir(exist_ok=True)

M_LIST = [3]    # <<< [3, 5, 8, 10]    
PROBLEM_TYPES = ["concave"] #<<< "convex","concave"

METRIC = "IGD"     # <<< "IGD", "COV"

SAMPLER_ORDER = ["SS", "PHS"]   # SS first, then PS

# ============================================================
# REGEX (extended to include Coverage Error)
# ============================================================
re_alpha = re.compile(r"^Problem:\s*alpha\s*=\s*([0-9.eE+-]+)\s*$")

re_row = re.compile(
    r"^(SS|PHS)\s*\|\s*refs=\s*(\d+)\s*\|\s*hits=\s*(\d+)\s*\|\s*"
    r"TruePF=\s*(\d+)\s*\|\s*"
    r"HV=([0-9.eE+-]+)\s*\|\s*"
    r"IGD=([0-9.eE+-]+)\s*\|\s*"
    r"IGD\+=([0-9.eE+-]+)\s*\|\s*"
    r"COV=([0-9.eE+-]+)\s*$"
)

# ============================================================
# PARSER
# ============================================================
def parse_results_alpha(M_target: int, problem_type: str) -> pd.DataFrame:
    fp = RESULTS_DIR / f"Results_{problem_type}_M{M_target}_seed_0.txt"
    if not fp.exists():
        raise FileNotFoundError(fp)

    rows = []
    current_alpha = None

    for raw in fp.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line:
            continue

        ma = re_alpha.match(line)
        if ma:
            current_alpha = float(ma.group(1))
            continue

        mr = re_row.match(line)
        if mr and current_alpha is not None:
            rows.append({
                "M": M_target,
                "problem_type": problem_type,
                "alpha": current_alpha,
                "sampler": mr.group(1),
                "refs": int(mr.group(2)),
                "hits": int(mr.group(3)),
                "TruePF": int(mr.group(4)),
                "HV": float(mr.group(5)),
                "IGD": float(mr.group(6)),
                "IGD+": float(mr.group(7)),
                "COV": float(mr.group(8)),
            })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No data parsed from {fp.name}")

    return df

# ============================================================
# TABLE BUILDER (ALL M COMBINED)
# ============================================================
def build_combined_table(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    if metric not in {"HV", "IGD", "IGD+", "COV"}:
        raise ValueError("metric must be 'HV', 'IGD', 'IGD+', or 'COV'")

    # Enforce sampler order
    df["sampler"] = pd.Categorical(
        df["sampler"],
        categories=SAMPLER_ORDER,
        ordered=True
    )

    table = (
        df
        .pivot_table(
            index=["M", "refs"],
            columns=["alpha", "sampler"],
            values=metric
        )
        .sort_index(axis=0)     # M → refs
        .sort_index(axis=1)     # alpha → sampler
    )

    # Clean column headers
    table.columns = pd.MultiIndex.from_tuples(
        [
            (int(a) if float(a).is_integer() else a, s)
            for a, s in table.columns
        ],
        names=["alpha", "sampler"]
    )

    table.index.names = ["M", "Ref. points"]
    return table

# ============================================================
# MAIN
# ============================================================
all_dfs = []

for M in M_LIST:
    for problem_type in PROBLEM_TYPES:
        try:
            df = parse_results_alpha(M, problem_type)
            all_dfs.append(df)
        except Exception as e:
            print(f"[Skip] M={M}, type={problem_type}: {e}")

if not all_dfs:
    raise RuntimeError("No data loaded")

df_all = pd.concat(all_dfs, ignore_index=True)

final_table = build_combined_table(df_all, METRIC)

final_table = final_table.round(6)

# Save
out_csv = OUT_DIR / f"{METRIC}_{PROBLEM_TYPES[0]}_table_ALL_M.csv"
out_xlsx = OUT_DIR / f"{METRIC}_{PROBLEM_TYPES[0]}_table_ALL_M.xlsx"

final_table.to_csv(out_csv)
final_table.to_excel(out_xlsx)

print(f"[Saved] {out_csv}")
print(f"[Saved] {out_xlsx}")


In [ ]:
##########################################################################################
'''Statistical Analysis: Significance check and detailed tables (SS vs PHS):
Significance check is performed by comparing the SS value against the mean ± std of PHS.
Comparison results are categorized as "SS better", "PHS better", or "similar" based on 
whether SS is outside the mean ± std range of PHS.
The final tables include both the formatted mean ± std for PHS and the raw SS values,
along with the comparison result for each (M, alpha, refs) configuration.
The detailed table contains all raw data for further analysis or reporting.'''
##########################################################################################

import re
from pathlib import Path
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
RESULTS_DIR = Path("Results")
OUT_DIR = Path("Results/Tables")
OUT_DIR.mkdir(exist_ok=True)

M_LIST = [3]
PROBLEM_TYPES = ["concave"]  #<<< "convex","concave"

METRIC = "IGD"   # <<<"IGD", "COV"

# Metric direction
LOWER_IS_BETTER = METRIC in ["IGD", "IGD+", "COV"]

# ============================================================
# REGEX
# ============================================================
re_alpha = re.compile(r"^Problem:\s*alpha\s*=\s*([0-9.eE+-]+)\s*$")

re_row = re.compile(
    r"^(SS|PHS)\s*\|\s*refs=\s*(\d+)\s*\|\s*hits=\s*(\d+)\s*\|\s*"
    r"TruePF=\s*(\d+)\s*\|\s*"
    r"HV=([0-9.eE+-]+)\s*\|\s*"
    r"IGD=([0-9.eE+-]+)\s*\|\s*"
    r"IGD\+=([0-9.eE+-]+)\s*\|\s*"
    r"COV=([0-9.eE+-]+)\s*$"
)

re_seed = re.compile(r"seed[_=]?(\d+)", re.IGNORECASE)

# ============================================================
# PARSER
# ============================================================
def parse_file(fp: Path, M_target: int, problem_type: str):
    m = re_seed.search(fp.name)
    if not m:
        raise ValueError(f"No seed in filename: {fp.name}")

    seed = int(m.group(1))

    rows = []
    current_alpha = None

    for line in fp.read_text(errors="ignore").splitlines():
        line = line.strip()
        if not line:
            continue

        ma = re_alpha.match(line)
        if ma:
            current_alpha = float(ma.group(1))
            continue

        mr = re_row.match(line)
        if mr and current_alpha is not None:
            rows.append({
                "problem_type": problem_type,
                "seed": seed,
                "M": M_target,
                "alpha": current_alpha,
                "sampler": mr.group(1),
                "refs": int(mr.group(2)),
                "value": float(mr.group(
                    {"HV":5,"IGD":6,"IGD+":7,"COV":8}[METRIC]
                )),
            })

    return pd.DataFrame(rows)

# ============================================================
# LOAD DATA
# ============================================================
dfs = []

for M in M_LIST:
    for ptype in PROBLEM_TYPES:
        pattern = f"Results_{ptype}_M{M}_seed*.txt"
        for fp in RESULTS_DIR.glob(pattern):
            dfs.append(parse_file(fp, M, ptype))

df = pd.concat(dfs, ignore_index=True)

# ============================================================
# SPLIT SS vs PS
# ============================================================
ss = df[df["sampler"] == "SS"].drop_duplicates(
    ["M", "alpha", "refs"]
)

ps = df[df["sampler"] == "PHS"]

# ============================================================
# PS stats
# ============================================================
ps_stats = (
    ps.groupby(["M", "alpha", "refs"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# ============================================================
# MERGE
# ============================================================
merged = pd.merge(
    ss[["M", "alpha", "refs", "value"]],
    ps_stats,
    on=["M", "alpha", "refs"],
    how="inner"
)

merged.rename(columns={"value": "SS"}, inplace=True)

# ============================================================
# SIGNIFICANCE CHECK
# ============================================================
def compare(row):
    ss = row["SS"]
    mean = row["mean"]
    std = row["std"]

    if LOWER_IS_BETTER:
        if ss < mean - std:
            return "SS better"
        elif ss > mean + std:
            return "PHS better"
    else:
        if ss > mean + std:
            return "SS better"
        elif ss < mean - std:
            return "PHS better"

    return "similar"

merged["result"] = merged.apply(compare, axis=1)

# ============================================================
# FORMAT TABLE
# ============================================================
merged["PS"] = merged.apply(
    lambda r: f"{r['mean']:.6f} ± {r['std']:.6f}",
    axis=1
)

merged["SS"] = merged["SS"].map(lambda x: f"{x:.6f}")

table = merged.pivot_table(
    index=["M", "refs"],
    columns="alpha",
    values=["SS", "PHS", "result"],
    aggfunc="first"
)

# ============================================================
# SAVE
# ============================================================
table.to_csv(OUT_DIR / f"Statistical_results_{METRIC}_SS_vs_PHS_{PROBLEM_TYPES[0]}.csv")
merged.to_csv(OUT_DIR / f"Statistical_results_{METRIC}_detailed_{PROBLEM_TYPES[0]}.csv", index=False)


In [ ]:
# ============================================================
# DTLZ7 (Discontinuous front problem)
# SS vs PS — Intersections, HV, IGD, IGD+
# ============================================================
#Import packages
import os
import numpy as np
from numba import njit, prange
import itertools
import math
from pathlib import Path
import plotly.graph_objects as go
import matplotlib.pyplot as plt

#Import utilities
from Codes.PHS_sampler import phs_sampler
from Codes.hv_qmc import Compute_HV

# ============================================================
#  Output directories
# ============================================================
BASE_DIR = Path.cwd()
FIG_DIR = BASE_DIR / "Figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = BASE_DIR / "Results_disconnected"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
#  Das–Dennis
# ============================================================
def das_dennis(M, H):
    slots = H + M - 1
    pts = []
    for c in itertools.combinations(range(slots), M - 1):
        prev = -1
        counts = []
        for pos in c:
            counts.append(pos - prev - 1)
            prev = pos
        counts.append(slots - prev - 1)
        w = np.array(counts, float) / H
        w /= w.sum()
        pts.append(w)
    return np.vstack(pts)


# ============================================================
#  Shift to sum=M hyperplane
# ============================================================
def shift_to_hyperplane(pts, target_sum):
    pts = np.asarray(pts, float)
    alphas = (target_sum - pts.sum(axis=1)) / pts.shape[1]
    return pts + alphas[:, None]

# ============================================================
# DTLZ7 PF (original space)
# ============================================================
def dtlz7_pf_value(F_head, M):
    return 2.0 * M - np.sum(
        F_head * (1.0 + np.sin(3.0 * np.pi * F_head)),
        axis=-1
    )


def sample_true_pf(M, N=4000):
    assert M == 3, "This grid version is for M=3"

    n = int(np.sqrt(N))  # grid resolution

    f1 = np.linspace(0, 1, n)
    f2 = np.linspace(0, 1, n)

    F1, F2 = np.meshgrid(f1, f2)
    head = np.column_stack([F1.ravel(), F2.ravel()])

    fm = dtlz7_pf_value(head, M)
    PF = np.column_stack([head, fm])

    return PF

# ============================================================
# Normalization utilities
# ============================================================
def make_normalizer(F):
    fmin = F.min(axis=0)
    fmax = F.max(axis=0)
    scale = fmax - fmin
    scale[scale == 0] = 1.0
    return fmin, scale

def normalize(F, fmin, scale):
    return (F - fmin) / scale

def unnormalize(Fn, fmin, scale):
    return fmin + Fn * scale


# ============================================================
# ND filtering
# ============================================================
def nondominated(F):
    N = len(F)
    keep = np.ones(N, dtype=bool)
    for i in range(N):
        if not keep[i]:
            continue
        for j in range(N):
            if i == j or not keep[j]:
                continue
            if np.all(F[j] <= F[i]) and np.any(F[j] < F[i]):
                keep[i] = False
                break
    return F[keep]

# ============================================================
# DTLZ7 PF normalized
# ============================================================
def dtlz7_pf_value_normalized(head_n, M, fmin, scale):
    head_orig = unnormalize(head_n, fmin[:M-1], scale[:M-1])
    fm_orig = dtlz7_pf_value(head_orig, M)
    fm_n = (fm_orig - fmin[M-1]) / scale[M-1]
    return fm_n


# ============================================================
# INTERSECTION (normalized space)
# ============================================================
def intersect_pf(Pn, M, fmin, scale, n_grid=2000):
    results = []

    for p in Pn:

        t_lo = max(0.0, np.max(p - 1.0))
        t_hi = np.min(p)

        if t_lo > t_hi:
            continue

        ts = np.linspace(t_lo, t_hi, n_grid)
        F = p[None, :] - ts[:, None]

        head = F[:, :M-1]
        fm_ray = F[:, M-1]
        fm_pf = dtlz7_pf_value_normalized(head, M, fmin, scale)

        phi = fm_ray - fm_pf

        # find sign change
        for i in range(len(ts) - 1):
            if phi[i] * phi[i+1] < 0:

                lo, hi = ts[i], ts[i+1]

                for _ in range(50):
                    mid = 0.5 * (lo + hi)
                    head_mid = (p[:M-1] - mid)[None, :]
                    fm_mid = dtlz7_pf_value_normalized(head_mid, M, fmin, scale)[0]
                    val = (p[M-1] - mid) - fm_mid

                    if abs(val) < 1e-10:
                        break

                    if phi[i] * val < 0:
                        hi = mid
                    else:
                        lo = mid

                t = 0.5 * (lo + hi)
                X = p - t
                results.append(X)
                break

    if len(results) == 0:
        return np.empty((0, M))

    return np.array(results)

# ============================================================
#  Metrics - HV, IGD, IGD+, Coverage error
# ============================================================
def safe_hv(A):
    if len(A) == 0:
        return float("nan")
    return float(Compute_HV(A))

@njit(parallel=True, fastmath=True)
def _min_dists_igd(A, P):
    nP, M = P.shape
    nA = A.shape[0]
    out = np.empty(nP, dtype=np.float64)

    for i in prange(nP):
        min_d2 = 1e300
        for j in range(nA):
            d2 = 0.0
            for k in range(M):
                diff = P[i, k] - A[j, k]
                d2 += diff * diff
            if d2 < min_d2:
                min_d2 = d2
        out[i] = np.sqrt(min_d2)
    return out

def igd(A, P):
    if len(A) == 0 or len(P) == 0:
        return float("nan")
    A = np.asarray(A, dtype=np.float64)
    P = np.asarray(P, dtype=np.float64)
    mins = _min_dists_igd(A, P)
    return float(mins.mean())



@njit(parallel=True, fastmath=True)
def _min_dists_igd_plus(A, P):
    nP, M = P.shape
    nA = A.shape[0]
    out = np.empty(nP, dtype=np.float64)

    for i in prange(nP):
        min_d2 = 1e300
        for j in range(nA):
            d2 = 0.0
            for k in range(M):
                diff = A[j, k] - P[i, k]
                if diff > 0.0:
                    d2 += diff * diff
            if d2 < min_d2:
                min_d2 = d2
        out[i] = np.sqrt(min_d2)
    return out

def igd_plus(A, P):
    if len(A) == 0 or len(P) == 0:
        return float("nan")
    A = np.asarray(A, dtype=np.float64)
    P = np.asarray(P, dtype=np.float64)
    mins = _min_dists_igd_plus(A, P)
    return float(mins.mean())


@njit(parallel=True, fastmath=True)
def _min_dists_for_coverage(A, P):
    nP, M = P.shape
    nA = A.shape[0]
    out = np.empty(nP, dtype=np.float64)

    for i in prange(nP):
        min_d2 = 1e300
        for j in range(nA):
            d2 = 0.0
            for k in range(M):
                diff = P[i, k] - A[j, k]
                d2 += diff * diff
            if d2 < min_d2:
                min_d2 = d2
        out[i] = np.sqrt(min_d2)
    return out

def coverage_error(A, P):
    if len(A) == 0 or len(P) == 0:
        return float("nan")
    A = np.asarray(A, dtype=np.float64)
    P = np.asarray(P, dtype=np.float64)
    mins = _min_dists_for_coverage(A, P)
    return float(mins.max())

# ============================================================
#  Plotting (M=3)
# ============================================================
def plot_3d(PF, P_ref, I_pts, title):
    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=PF[:,0], y=PF[:,1], z=PF[:,2],
        mode="markers",
        marker=dict(size=4, opacity=0.15, color="blue"),
        name="True PF"
    ))

    fig.add_trace(go.Scatter3d(
        x=P_ref[:,0], y=P_ref[:,1], z=P_ref[:,2],
        mode="markers",
        marker=dict(size=4, color="red"),
        name="Refs"
    ))

    if len(I_pts):
        fig.add_trace(go.Scatter3d(
            x=I_pts[:,0], y=I_pts[:,1], z=I_pts[:,2],
            mode="markers",
            marker=dict(size=6, color="green"),
            name="Intersections"
        ))

    fig.update_layout(title=title)
    fig.update_layout(
        scene=dict(
            aspectmode="cube",
            xaxis=dict(
                title=dict(text="F1", font=dict(size=22)),
                tickfont=dict(size=16),
            ),
            yaxis=dict(
                title=dict(text="F2", font=dict(size=22)),
                tickfont=dict(size=16),
            ),
            zaxis=dict(
                title=dict(text="F3", font=dict(size=22)),
                tickfont=dict(size=16),
            ),
        ),
        margin=dict(l=0, r=0, t=0, b=0),
        width=900,
        height=800,
    )

    fig.show()

def save_matplotlib_3d(PF, P_ref, I_pts, title, fname):

    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(*PF.T, s=18, alpha=0.15, c="blue", marker="o", label = "True PF")
    # ax.scatter(*P_ref.T, alpha=0.75, s=18, c="red", label="Refs") 
    if len(I_pts):
        ax.scatter(*I_pts.T, s=25, c="green",  marker="o", label="Intersections")

    ax.view_init(elev=25, azim=45)
    ax.set_box_aspect((1, 1, 1))
    ax.margins(0)
    plt.subplots_adjust(left=0.1, right=1, bottom=0.05, top=1)
    ax.set_aspect('equal')
    # Axis labels + tick size
    ax.set_xlabel(r'$f1$', fontsize=15, labelpad=6)
    ax.set_ylabel(r'$f2$', fontsize=15, labelpad=6)
    ax.set_zlabel(r'$f3$', fontsize=15, labelpad=6)
    ax.legend(loc="upper right", fontsize=12)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.grid(False)
    out = FIG_DIR / fname
    fig.savefig(out, dpi=300, bbox_inches="tight", pad_inches=0.2)
    plt.close(fig)

    print(f"[Saved] {out}")

# ============================================================
#  Plotting (M>3) - Higher dimensions
# ============================================================

def parallel_axis_plot(samples,
                       nlines=300,
                       normalize=False,
                       color_by=None,
                       axis_labels=None,
                       alpha=0.5,
                       linewidth=0.8,
                       cmap="viridis",
                       figsize=(12, 6),
                       seed=0,
                       title=None,
                       show=True,
                       save_figure=False,
                       save_path="Results"):
    """
    Parallel-axis (parallel-coordinates) plot for samples.

    Parameters
    ----------
    samples : ndarray, shape (n_samples, N)
        Input samples (RAW coordinates, e.g. loaded from .npy).
    nlines : int
        Number of sample lines to plot (random subset if samples are many).
    normalize : bool
        If True, normalize each axis independently to [0,1].
        If False, plot raw coordinates.
    color_by : None or int
        If None, all lines are same color.
        If int k, color lines by value on axis k.
    axis_labels : list of str or None
        Labels for axes. Defaults to x0, x1, ..., x{N-1}.
    """

    samples = np.asarray(samples)
    if samples.ndim != 2:
        raise ValueError("samples must be a 2D array (n_samples, N)")

    n_samples, N = samples.shape

    # subsample lines
    rng = np.random.default_rng(seed)
    if n_samples > nlines:
        idx = rng.choice(n_samples, size=nlines, replace=False)
        X = samples[idx]
    else:
        X = samples.copy()

    # normalization (optional)
    if normalize:
        mn = X.min(axis=0)
        mx = X.max(axis=0)
        scale = mx - mn
        scale[scale == 0.0] = 1.0
        Xp = (X - mn) / scale
        y_label = "Normalized value"
    else:
        Xp = X
        y_label = "Raw coordinate value"

    # axis labels
    if axis_labels is None:
        axis_labels = [f"x{i}" for i in range(N)]

    xs = np.arange(N)

    # color mapping
    if color_by is None:
        colors = [(0, 0, 0, alpha)] * Xp.shape[0]
    else:
        if not (0 <= color_by < N):
            raise ValueError("color_by must be None or an axis index in [0, N-1]")
        vals = Xp[:, color_by]
        vmin, vmax = vals.min(), vals.max()
        if vmax > vmin:
            vals = (vals - vmin) / (vmax - vmin)
        cmap_obj = plt.get_cmap(cmap)
        colors = [cmap_obj(v) for v in vals]

    # plotting
    fig, ax = plt.subplots(figsize=figsize)
    for i in range(Xp.shape[0]):
        ax.plot(xs, Xp[i], color=colors[i], alpha=alpha, linewidth=linewidth)

    # vertical axes
    for xi in xs:
        ax.axvline(xi, color="k", linewidth=0.8, alpha=0.5)

    ax.set_xlim(xs[0], xs[-1])
    ax.set_xticks(xs)
    ax.set_xticklabels(axis_labels, rotation=45, ha="right")
    ax.set_ylabel(y_label)

    if title:
        ax.set_title(title)

    plt.tight_layout()

    if save_figure:
        plt.savefig(save_path, dpi=300)

    if show:
        plt.show()

    return fig, ax

# ============================================================
#  main
# ============================================================
def run(
    M=3,
    samplers=("SS", "PHS"),
    H_list=(15,),
    seed=0,
    save_figures=True,
):
    
    results_path = RESULTS_DIR / f"Results_M{M}_seed_{seed}.txt"

    # ------------------------------------------------------------------
    # Cache directories
    # ------------------------------------------------------------------
    ref_root = RESULTS_DIR / "Ref_points"
    int_root = RESULTS_DIR / "Intersections"

    for sampler in samplers:
        (ref_root / sampler).mkdir(parents=True, exist_ok=True)
        (int_root / sampler).mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------------
    # Main experiment
    # ------------------------------------------------------------------
    with open(results_path, "w") as f:
        f.write("============================================================\n")
        f.write(f"Alpha-only PF Experiment | type=None\n")
        f.write(f"M={M}, alphas= [2], seed={seed}\n")
        f.write("============================================================\n\n")

        for alpha in [2]:
            f.write(f"\n##############################\n")
            f.write(f"Problem: alpha = 2\n")
            f.write(f"##############################\n\n")

            PF_true = sample_true_pf(M)
            PF_true = nondominated(PF_true)
                # --- NORMALIZATION ---
            fmin, scale = make_normalizer(PF_true)
            PF_n = normalize(PF_true, fmin, scale)

            for H in H_list:
                f.write(f"--- H={H} ---\n")

                for sampler in samplers:

                    # ==================================================
                    # Reference points
                    # ==================================================
                    if sampler == "SS":
                        ref_file = (
                            ref_root / sampler /
                            f"ref_M{M}_H{H}.npy"
                        )
                    else:  # PS (Sobol)
                        ref_file = (
                            ref_root / sampler /
                            f"ref_M{M}_H{H}_seed{seed}.npy"
                        )

                    if ref_file.exists():
                        P_ref = np.load(ref_file)
                    else:
                        if sampler == "SS":
                            P_ref = das_dennis(M, H)
                        else:
                            N_ref = math.comb(H + M - 1, M - 1)
                            P_ref = phs_sampler(
                                d=M,
                                N_target=N_ref,
                                seed=seed
                            )

                        P_ref = shift_to_hyperplane(
                            P_ref, target_sum=M
                        )
                        np.save(ref_file, P_ref)

                    # ==================================================
                    # Intersection points
                    # ==================================================
                    if sampler == "SS":
                        int_file = (
                            int_root / sampler /
                            f"int_M{M}_alpha{alpha}_H{H}.npy"
                        )
                    else:
                        int_file = (
                            int_root / sampler /
                            f"int_M{M}_alpha{alpha}_H{H}_seed{seed}.npy"
                        )

                    if int_file.exists():
                        I_pts = np.load(int_file)
                    else:
                        I_pts = intersect_pf(
                            P_ref, M, fmin, scale)
                            # --- ND filter ---
                        I_pts = nondominated(I_pts)
                        np.save(int_file, I_pts)

                    # ==================================================
                    # Metrics
                    # ==================================================
                    hv = safe_hv(I_pts)
                    igd_v = igd(I_pts, PF_n)
                    igdp_v = igd_plus(I_pts, PF_n)
                    cov_v = coverage_error(I_pts, PF_n)


                    print("================================")
                    print(f"type= None | alpha=2 | {sampler} | H={H}")
                    print(f"Refs={len(P_ref)} Hits={len(I_pts)} PF_True={len(PF_true)}")
                    print(f"HV={hv:.6f} IGD={igd_v:.6f} IGD+={igdp_v:.6f} COV={cov_v:.6f}")
                    print("================================")

                    f.write(
                        f"{sampler:2s} | refs={len(P_ref):6d} | hits={len(I_pts):6d} | TruePF={len(PF_true):6d} | "
                        f"HV={hv:.10f} | IGD={igd_v:.10f} | IGD+={igdp_v:.10f} | "
                        f"COV={cov_v:.10f}\n"
                    )

                    # ==================================================
                    # Plotting (unchanged)
                    # ==================================================
                    title = f"α=2 | {sampler} | M={M} H={H}"

                    if save_figures:
                        if M == 3:
                            plot_3d(PF_n, P_ref, I_pts, title)

                            save_matplotlib_3d(
                                PF_n,
                                P_ref,
                                I_pts,
                                title,
                                fname=f"alpha2_{sampler}_M{M}_H{H}"
                            )
                        else:
                            parallel_axis_plot(
                                P_ref,
                                title=f"Refs | {title}"
                            )
                            if len(I_pts):
                                parallel_axis_plot(
                                    I_pts,
                                    title=f"Intersections | {title}",
                                    save_figure=False,
                                    save_path=FIG_DIR / f"alpha2_{sampler}_M{M}_H{H}_intersections.png"
                                )

                f.write("\n")

        f.write("============================================================\n")

    print(f"\n[Saved results] {results_path}")
    return P_ref, I_pts

# ============================================================
#  RUN EXPERIMENTS
# ============================================================
if __name__ == "__main__":
    # ============================================================
    #Parameter settings for the experiments. samplers, H_list, seed, and save_figures as needed.
    # M = 3
    #samplers = ["SS", "PHS"]
    # H_list = [12, 15, 20, 25, 30] <<< M=3
    # Alpha value just set to 2 (it could be any value) for using the same post processing code

    # Experiments
    # ============================================================
    run(
        M=3,
        samplers=["SS", "PHS"],
        H_list=[12,15,20,25,30,50],
        seed=0,
        save_figures=True,
    )
    

In [ ]:
##################################################################
'''IGD and Coverage error are used jointly to assess convergence 
accuracy and Pareto-front coverage quality. A lower IGD value 
indicates that the approximated front is closer to the true Pareto
front, while a lower coverage error indicates better coverage
of the true Pareto front by the approximated front.'''

'''It read the results from the text files generated by the experiments 
from the Result folder, extracts the relevant data, and creates plots to
visualize the performance of the samplers (SS and PS) across different 
alpha values and reference point counts. The plots will show how the IGD 
and Coverage error metrics vary with the number of reference points for 
each sampler and alpha value, allowing for a comparative analysis of 
their performance in approximating the Pareto front.'''

##################################################################
#Plot results
import re
import math
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIG — CHANGE ONLY THESE
# ============================================================
RESULTS_DIR = Path("Results_disconnected")
FIG_DIR = Path("Figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

M_LIST = [3] 
PROBLEM_TYPES = ["concave"]

METRIC = "HV"      # <<< "IGD", "COV", "HV"
COLS_PER_ROW = 1

COLORS = {"SS": "red", "PHS": "green"}
MARKERS = {"SS": "o", "PHS": "s"}

# ============================================================
# REGEX (extended to include Coverage Error)
# ============================================================
re_alpha = re.compile(r"^Problem:\s*alpha\s*=\s*([0-9.eE+-]+)\s*$")
re_row = re.compile(
    r"^(SS|PHS)\s*\|\s*refs=\s*(\d+)\s*\|\s*hits=\s*(\d+)\s*\|\s*"
    r"TruePF=\s*(\d+)\s*\|\s*"
    r"HV=([0-9.eE+-]+)\s*\|\s*"
    r"IGD=([0-9.eE+-]+)\s*\|\s*"
    r"IGD\+=([0-9.eE+-]+)\s*\|\s*"
    r"COV=([0-9.eE+-]+)\s*$"
)

# ============================================================
# PARSER
# ============================================================
def parse_results_alpha(M_target: int, problem_type: str) -> pd.DataFrame:
    fp = RESULTS_DIR / f"Results_M{M_target}_seed_0.txt"
    if not fp.exists():
        raise FileNotFoundError(fp)

    rows = []
    current_alpha = None

    for raw in fp.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line:
            continue

        ma = re_alpha.match(line)
        if ma:
            current_alpha = float(ma.group(1))
            continue

        mr = re_row.match(line)
        if mr and current_alpha is not None:
            rows.append({
                "M": M_target,
                "problem_type": problem_type,
                "alpha": current_alpha,
                "sampler": mr.group(1),
                "refs": int(mr.group(2)),
                "hits": int(mr.group(3)),
                "TruePF": int(mr.group(4)),
                "HV": float(mr.group(5)),
                "IGD": float(mr.group(6)),
                "IGD+": float(mr.group(7)),
                "COV": float(mr.group(8)),
            })


    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No data parsed from {fp.name}")

    # sanity check
    dup = df.duplicated(subset=["alpha", "sampler", "refs"], keep=False)
    if dup.any():
        raise RuntimeError(
            f"Duplicate entries detected in {fp.name}:\n{df[dup]}"
        )

    return df

# ============================================================
# PLOTTING
# ============================================================
def plot_metric_by_alpha(
    df: pd.DataFrame,
    M_target: int,
    problem_type: str,
    metric: str,
    cols: int = 3,
):
    if metric not in {"HV", "IGD", "IGD+", "COV"}:
        raise ValueError("metric must be 'HV', 'IGD', 'IGD+', or 'COV'")

    alphas = sorted(df["alpha"].unique())
    n = len(alphas)
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(5.6 * cols, 4.4 * rows),
        sharey=False
    )

    # normalize axes indexing
    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]

    for i, alpha in enumerate(alphas):
        r = i // cols
        c = i % cols
        ax = axes[r][c]

        d = df[df["alpha"] == alpha]

        for sampler in ["SS", "PHS"]:
            ds = d[d["sampler"] == sampler].sort_values("refs")
            if ds.empty:
                continue

            ax.plot(
                ds["refs"],
                ds[metric],
                color=COLORS[sampler],
                marker=MARKERS[sampler],
                linewidth=2,
                markersize=6,
                label=sampler
            )

        ax.set_title(f"alpha = {alpha:g}")
        ax.set_xlabel("Reference points")
        ax.set_ylabel(metric)
        ax.grid(True, linestyle=":")
        ax.legend(frameon=False)

    # disable unused axes
    for j in range(n, rows * cols):
        r = j // cols
        c = j % cols
        axes[r][c].axis("off")

    fig.suptitle(
        f"{metric} vs reference points | M={M_target}",
        fontsize=14
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    out = FIG_DIR / f"{metric}_grid_M{M_target}_DTLZ7.png"
    fig.savefig(out, dpi=300)
    print(f"[Saved] {out}")

    plt.show()

# ============================================================
# MAIN
# ============================================================
for M in M_LIST:
    for problem_type in PROBLEM_TYPES:
        try:
            df = parse_results_alpha(M, problem_type)
            plot_metric_by_alpha(
                df,
                M_target=M,
                problem_type=problem_type,
                metric=METRIC,
                cols=COLS_PER_ROW,
            )
        except Exception as e:
            print(f"[Skip] M={M}, type={problem_type}: {e}")
